# GitHub Developer Collaboration Network & Link Prediction
## Academic Social Network Analysis (SNA) Notebook

This notebook demonstrates the end-to-end pipeline:
1. **Phase 1 & 2:** Configuration, GitHub Archive ingestion & Schema Understanding
2. **Phase 3 & 4:** Direct Dyadic Interaction Extraction & Ego-Network Construction
3. **Phase 5:** Data Preparation, Cleaning & Zero-Leakage Train/Test Splitting
4. **Phase 6:** Link Prediction Benchmark (CN, Jaccard, Adamic-Adar, PA) & Top-K Recommendations
5. **Phase 7:** Network Visualizations (Static, Ego, Predicted Links, PyVis Interactive, GraphML)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import networkx as nx
import pandas as pd
from src import load_config
from src.archive_collector import ArchiveCollector
from src.event_parser import EventParser
from src.graph_builder import GraphBuilder, compute_network_stats
from src.preprocessing import Preprocessor
from src.link_prediction import LinkPredictor
from src.evaluation import Evaluator
from src.recommendations import RecommendationEngine
from src.visualization import Visualizer

config = load_config()
print(f"Loaded configuration. Seed User: {config['github']['username']}")

### Step 1: Load Cached Archive & Parse Direct Interactions
We parse verified direct interactions (PR reviews, inline review comments, issue discussions) with full provenance.

In [ ]:
collector = ArchiveCollector(config)
archive_paths = collector.list_cached()
if not archive_paths:
    archive_paths = collector.collect(limit=1)

parser = EventParser(config)
interactions_df = parser.parse_files(archive_paths)
print(f"Parsed {len(interactions_df)} direct interactions from {len(archive_paths)} archive file(s).")
interactions_df.head(5)

### Step 2: Build Collaboration Network & Ego Expansion

In [ ]:
# Use top interacting user in the archive if seed user has sparse public slice
top_active_user = interactions_df['source_login'].value_counts().index[1]
builder = GraphBuilder(config)
builder._seed_user = top_active_user
builder._max_depth = 2

G, status = builder.build_network(interactions_df)
print(f"Built Ego-Network for '{top_active_user}': {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
compute_network_stats(G)

### Step 3: Zero-Leakage Train-Test Split & Negative Sampling

In [ ]:
prep = Preprocessor(config)
G_train, train_pos, test_pos, test_neg, tables = prep.prepare_and_split(G)
print(f"Train edges: {len(train_pos)} | Test edges: {len(test_pos)} | Test negative non-edges: {len(test_neg)}")
print(f"Zero leakage check: {any(G_train.has_edge(u, v) for u, v in test_pos)} test edges in G_train")

### Step 4: Link Prediction Evaluation (CN, Jaccard, Adamic-Adar, PA)

In [ ]:
predictor = LinkPredictor(config)
evaluator = Evaluator(config)

all_candidates = test_pos + test_neg
scores = predictor.score_candidates(G_train, all_candidates)
eval_df = evaluator.evaluate(scores, test_pos, test_neg)
eval_df

### Step 5: Explainable Top-K Recommendations

In [ ]:
rec_engine = RecommendationEngine(config, predictor)
recs = rec_engine.generate_recommendations(G_train, seed_user=top_active_user)
recs['ensemble'].head(10)

### Step 6: Visualizations & Deliverables

In [ ]:
viz = Visualizer(config)
p_overall = viz.plot_overall_network(G, seed_user=top_active_user)
p_ego = viz.plot_ego_network(G, seed_user=top_active_user)

# Format top 10 predicted edges for visualizer
top_preds = [(r['seed_user'], r['candidate'], r['score']) for r in recs['raw'].get('common_neighbors', [])[:10]]
p_pred = viz.plot_top10_predicted_links(G_train, seed_user=top_active_user, predicted_edges=top_preds)
viz.interactive_network_html(G_train, seed_user=top_active_user, predicted_edges=top_preds)
viz.export_graphml(G, "notebook_export.graphml")
print("Visualizations saved to outputs/visualizations/")